# 🎬 Footage Retrieval Engine — Interactive Google Colab Demo

This notebook demonstrates the standalone **Footage Retrieval Engine** built according to `SPEC.md`:
1. **Pre-spend Deduplication**: Exact match on `(provider, source_id)` or `source_url` before downloading.
2. **Adaptive Scene Chunking**: PySceneDetect + Sliding Window for atomic retrievable units.
3. **X-CLIP Base Embedding**: Joint video-text multimodal embeddings (512 dimensions).
4. **Vector Indexing**: High-performance semantic indexing with Zilliz Cloud / Milvus.
5. **Retrieval API & Fine Localization**: Sub-second semantic search + frame-level sub-range localization.

In [ ]:
# @title 1. Verify GPU Accelerator and Install Dependencies
!nvidia-smi

!pip install -q \n    pydantic pydantic-settings sqlalchemy requests python-dotenv \n    opencv-python-headless scenedetect yt-dlp \n    torch transformers pillow pymilvus imagekitio

print("✓ All dependencies installed successfully.")

In [ ]:
# @title 2. Import Footage Engine and Initialize Environment
import os
import footage_engine as fe
from footage_engine.retrieval.models import SearchFilters

# Check active configuration
cfg = fe.get_settings()
print(f"Database URL: {cfg.DATABASE_URL}")
print(f"Storage Backend: {cfg.STORAGE_BACKEND} ({cfg.LOCAL_STORAGE_DIR})")
print(f"Embedding Model: {cfg.DEFAULT_EMBEDDING_MODEL}")
print(f"Embedding Device: {cfg.EMBEDDING_DEVICE}")

In [ ]:
# @title 3. Ingest Footage (Direct URLs & Pre-Spend Dedup)

# Sample public creative commons videos
sample_urls = [
    "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4",
    "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerEscapes.mp4",
    "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerFun.mp4",
]

print("Ingesting videos...")
items = fe.ingest_url_list(sample_urls, provider="demo_feed")
for item in items:
    print(f"  ✓ Ingested: {item.id} | Provider: {item.provider} | Status: {item.status.value}")

print("\nTesting Duplicate Ingestion...")
dedup_item = fe.ingest(source_url=sample_urls[0], provider="demo_feed")
print(f"  ✓ Duplicate check succeeded! Returned existing ID: {dedup_item.id} without re-downloading.")

In [ ]:
# @title 4. Run Batch Processor (Chunking + X-CLIP Embeddings + Vector Indexing)
processor = fe.BatchProcessor()
stats = processor.process_all_pending()
print(f"Batch processing finished: {stats}")

In [ ]:
# @title 5. Execute Semantic Search
query = "outdoor sports and running adventure" # @param {type:"string"}
top_k = 5 # @param {type:"integer"}

results = fe.search(query=query, top_k=top_k)

print(f"Search results for: '{query}'\n")
for i, res in enumerate(results, 1):
    print(f"[{i}] Score: {res.score:.4f} | Chunk ID: {res.chunk_id}")
    print(f"    Range: {res.start_ts:.2f}s - {res.end_ts:.2f}s (Duration: {res.duration_sec:.2f}s)")
    print(f"    Provider: {res.provider} | Usage count: {res.usage_count}")
    print(f"    URL: {res.storage_url}\n")

In [ ]:
# @title 6. Frame-Level Fine Localization
if results:
    winning_chunk_id = results[0].chunk_id
    print(f"Localizing peak activity within winning chunk: {winning_chunk_id}...")
    refined_start, refined_end = fe.fine_localize(winning_chunk_id, query=query)
    print(f"  ✓ Refined timeline window: [{refined_start:.2f}s - {refined_end:.2f}s]")